In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


# Attention Model Evaluation

Ноутбук для оценки обученной `AttentionPhonemeModel` по каждой фразе.

Считает:
- `CER` по каждой фразе
- ошибку определения границ в миллисекундах
- сохраняет результаты в CSV


In [ ]:
import csv
import os
from statistics import median

import numpy as np
import torch

from CTC_tools import AttentionPhonemeModel


## Configuration

Проверь пути к чекпоинту и данным перед запуском.

In [ ]:
CHECKPOINT_PATH = r"" + str(DEFAULT_CTC_ATTENTION_MODEL) + ""
ROOT_DIR = r"\\nid-tts-02\mnt\hot_store\trainee_common\ananeva\CORPRES_embs_no_averaging"
SPEAKERS = ["Galina", "Victoria", "Petr", "Alexander"]
OUTPUT_CSV = "attention_evaluation.csv"

FRAME_MS = 20.0
MAX_STEPS = 512
STOP_THRESHOLD = 0.5
DECODE_METHOD = "argmax"  # "center_of_mass" or "argmax"

HIDDEN_DIM = 256
NUM_LAYERS = 2
PHONEME_EMB_DIM = 128
DECODER_DIM = 256
ATTENTION_DIM = 128

MAX_SEQUENCES_PER_SPEAKER = 1000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
PHONEME_LIST_FULL = [
    "a0", "a1", "a2", "a4", "b", "b'", "c", "ch", "ch_", "d", "d'",
    "e0", "e1", "e2", "e4", "f", "f'", "g", "g'", "h", "h'", "i0", "i1",
    "i2", "i4", "j", "jr", "jl", "ji4", "k", "k'", "l", "l'", "m", "m'",
    "n", "n'", "o0", "o1", "o2", "o4", "p", "p'", "r", "r'", "s", "s'",
    "sc", "sh", "t", "t'", "u0", "u1", "u2", "u4", "v", "v'", "y0", "y1",
    "y2", "y4", "z", "z'", "zh", "zh'", "C", "CH", "H", "SC",
]
PHONEME_LIST = PHONEME_LIST_FULL + ["sil"]
IDX2PHONEME = {idx + 1: phoneme for idx, phoneme in enumerate(PHONEME_LIST)}

print("num_phonemes:", len(PHONEME_LIST))


## Helpers

In [ ]:
def load_dataset_with_metadata(root_dir, speakers, phoneme_list, max_sequences_per_speaker=None):
    samples = []

    for speaker in speakers:
        speaker_root_dir = os.path.join(root_dir, speaker)
        speaker_seq_count = 0

        for root, _, files in os.walk(speaker_root_dir):
            if max_sequences_per_speaker is not None and speaker_seq_count >= max_sequences_per_speaker:
                break

            for file_name in files:
                if max_sequences_per_speaker is not None and speaker_seq_count >= max_sequences_per_speaker:
                    break

                if not file_name.endswith("_no_av_ph.txt"):
                    continue

                utterance_id = file_name.replace("_no_av_ph.txt", "")
                phoneme_file_path = os.path.join(root, file_name)
                emb_file_path = os.path.join(root, utterance_id + "_no_av_embs.npy")

                if not os.path.exists(emb_file_path):
                    continue

                with open(phoneme_file_path, "r", encoding="utf-8") as f:
                    raw_phonemes = [line.strip() for line in f.readlines()]

                emb_array = np.load(emb_file_path)
                if len(raw_phonemes) != len(emb_array):
                    continue

                filtered_embeddings = []
                filtered_frame_phonemes = []
                for raw_phoneme, emb in zip(raw_phonemes, emb_array):
                    parts = raw_phoneme.split("_")
                    if len(parts) != 3:
                        continue

                    base_phoneme = parts[1]
                    if base_phoneme not in phoneme_list:
                        continue

                    filtered_embeddings.append(emb)
                    filtered_frame_phonemes.append(base_phoneme)

                if not filtered_frame_phonemes:
                    continue

                collapsed_phonemes = []
                boundary_frames = []
                for frame_idx, phoneme in enumerate(filtered_frame_phonemes):
                    if not collapsed_phonemes or phoneme != collapsed_phonemes[-1]:
                        collapsed_phonemes.append(phoneme)
                        boundary_frames.append(frame_idx)

                samples.append(
                    {
                        "speaker": speaker,
                        "utterance_id": utterance_id,
                        "phoneme_file_path": phoneme_file_path,
                        "emb_file_path": emb_file_path,
                        "embedding": torch.tensor(np.stack(filtered_embeddings), dtype=torch.float32),
                        "frame_phonemes": filtered_frame_phonemes,
                        "target_phonemes": collapsed_phonemes,
                        "target_boundary_frames": boundary_frames,
                    }
                )
                speaker_seq_count += 1

        print(f"{speaker}: loaded {speaker_seq_count} phrases")

    return samples


def attention_step_times(attention, input_lengths, method="center_of_mass", frame_ms=20.0):
    batch_size, _, max_input_len = attention.shape
    frame_positions = torch.arange(max_input_len, device=attention.device, dtype=attention.dtype)
    all_times_ms = []

    for batch_idx in range(batch_size):
        input_len = int(input_lengths[batch_idx].item())
        attn = attention[batch_idx, :, :input_len]

        if method == "argmax":
            frame_idx = attn.argmax(dim=-1).float()
        elif method == "center_of_mass":
            weights = attn / attn.sum(dim=-1, keepdim=True).clamp_min(1e-8)
            frame_idx = (weights * frame_positions[:input_len]).sum(dim=-1)
        else:
            raise ValueError("method must be 'argmax' or 'center_of_mass'")

        all_times_ms.append((frame_idx * frame_ms).detach().cpu().tolist())

    return all_times_ms


def greedy_decode_attention(model, x_single, idx2phoneme, device, max_steps=512, start_idx=0, frame_ms=20.0, method="center_of_mass", stop_threshold=0.5):
    model.eval()

    with torch.no_grad():
        x = x_single.unsqueeze(0).to(device)
        input_lengths = torch.tensor([x_single.size(0)], device=device)

        encoder_outputs = model.encode(x)
        encoder_proj = model.encoder_proj(encoder_outputs)

        hidden = encoder_outputs.new_zeros(1, model.decoder_dim)
        cell = encoder_outputs.new_zeros(1, model.decoder_dim)
        context = encoder_outputs.new_zeros(1, encoder_outputs.size(-1))
        current_token = torch.tensor([start_idx], device=device)

        pred_ids = []
        attention_steps = []
        stop_probs = []

        for _ in range(max_steps):
            embedded = model.phoneme_embedding(current_token)
            decoder_input = torch.cat([embedded, context], dim=-1)

            hidden, cell = model.decoder_cell(decoder_input, (hidden, cell))
            context, attn = model._attention(encoder_outputs, encoder_proj, hidden, input_lengths)

            decoder_state = torch.cat([hidden, context], dim=-1)
            token_logits = model.output_layer(decoder_state)
            stop_logit = model.stop_layer(decoder_state)
            stop_prob = torch.sigmoid(stop_logit).item()

            next_token = token_logits.argmax(dim=-1)
            token_id = int(next_token.item())

            if token_id == 0:
                break

            pred_ids.append(token_id)
            attention_steps.append(attn.squeeze(0))
            stop_probs.append(stop_prob)
            current_token = next_token

            if stop_prob >= stop_threshold:
                break

        if not pred_ids:
            return [], [], [], []

        attention_matrix = torch.stack(attention_steps, dim=0).unsqueeze(0)
        pred_times_ms = attention_step_times(
            attention_matrix,
            input_lengths=input_lengths,
            method=method,
            frame_ms=frame_ms,
        )[0]
        pred_phonemes = [idx2phoneme[idx] for idx in pred_ids]
        return pred_ids, pred_phonemes, pred_times_ms, stop_probs


def align_sequences(ref_tokens, hyp_tokens):
    ref_len = len(ref_tokens)
    hyp_len = len(hyp_tokens)

    dp = [[0] * (hyp_len + 1) for _ in range(ref_len + 1)]
    back = [[None] * (hyp_len + 1) for _ in range(ref_len + 1)]

    for i in range(1, ref_len + 1):
        dp[i][0] = i
        back[i][0] = "delete"
    for j in range(1, hyp_len + 1):
        dp[0][j] = j
        back[0][j] = "insert"

    for i in range(1, ref_len + 1):
        for j in range(1, hyp_len + 1):
            sub_cost = 0 if ref_tokens[i - 1] == hyp_tokens[j - 1] else 1
            choices = [
                (dp[i - 1][j] + 1, "delete"),
                (dp[i][j - 1] + 1, "insert"),
                (dp[i - 1][j - 1] + sub_cost, "equal" if sub_cost == 0 else "substitute"),
            ]
            dp[i][j], back[i][j] = min(choices, key=lambda item: item[0])

    i = ref_len
    j = hyp_len
    operations = []
    while i > 0 or j > 0:
        op = back[i][j]
        if op in {"equal", "substitute"}:
            operations.append((op, i - 1, j - 1))
            i -= 1
            j -= 1
        elif op == "delete":
            operations.append((op, i - 1, None))
            i -= 1
        elif op == "insert":
            operations.append((op, None, j - 1))
            j -= 1
        else:
            break

    operations.reverse()
    return dp[ref_len][hyp_len], operations


def cer_and_boundary_metrics(ref_tokens, hyp_tokens, ref_times_ms, hyp_times_ms):
    edit_distance, operations = align_sequences(ref_tokens, hyp_tokens)
    ref_len = max(len(ref_tokens), 1)

    substitutions = 0
    insertions = 0
    deletions = 0
    boundary_errors_ms = []

    for op, ref_idx, hyp_idx in operations:
        if op == "substitute":
            substitutions += 1
        elif op == "insert":
            insertions += 1
        elif op == "delete":
            deletions += 1
        elif op == "equal" and ref_idx > 0 and hyp_idx > 0:
            boundary_errors_ms.append(abs(ref_times_ms[ref_idx] - hyp_times_ms[hyp_idx]))

    return {
        "cer": edit_distance / ref_len,
        "substitutions": substitutions,
        "insertions": insertions,
        "deletions": deletions,
        "matched_boundaries": len(boundary_errors_ms),
        "mean_boundary_error_ms": sum(boundary_errors_ms) / len(boundary_errors_ms) if boundary_errors_ms else None,
        "median_boundary_error_ms": median(boundary_errors_ms) if boundary_errors_ms else None,
        "max_boundary_error_ms": max(boundary_errors_ms) if boundary_errors_ms else None,
    }


def load_model(checkpoint_path, input_dim, num_phonemes, device):
    model = AttentionPhonemeModel(
        input_dim=input_dim,
        hidden_dim=HIDDEN_DIM,
        num_phonemes=num_phonemes,
        num_layers=NUM_LAYERS,
        phoneme_emb_dim=PHONEME_EMB_DIM,
        decoder_dim=DECODER_DIM,
        attention_dim=ATTENTION_DIM,
    )
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model


## Run Evaluation

In [ ]:
samples = load_dataset_with_metadata(
    root_dir=ROOT_DIR,
    speakers=SPEAKERS,
    phoneme_list=PHONEME_LIST,
    max_sequences_per_speaker=MAX_SEQUENCES_PER_SPEAKER,
)

if not samples:
    raise RuntimeError("No samples found for evaluation.")

input_dim = samples[0]["embedding"].shape[1]
model = load_model(CHECKPOINT_PATH, input_dim=input_dim, num_phonemes=len(PHONEME_LIST), device=device)

results = []
for sample in samples:
    _, pred_phonemes, pred_times_ms, stop_probs = greedy_decode_attention(
        model=model,
        x_single=sample["embedding"],
        idx2phoneme=IDX2PHONEME,
        device=device,
        max_steps=MAX_STEPS,
        frame_ms=FRAME_MS,
        method=DECODE_METHOD,
        stop_threshold=STOP_THRESHOLD,
    )

    true_phonemes = sample["target_phonemes"]
    true_boundary_times_ms = [frame_idx * FRAME_MS for frame_idx in sample["target_boundary_frames"]]

    metrics = cer_and_boundary_metrics(
        ref_tokens=true_phonemes,
        hyp_tokens=pred_phonemes,
        ref_times_ms=true_boundary_times_ms,
        hyp_times_ms=pred_times_ms,
    )

    result_row = {
        "speaker": sample["speaker"],
        "utterance_id": sample["utterance_id"],
        "phoneme_file_path": sample["phoneme_file_path"],
        "emb_file_path": sample["emb_file_path"],
        "num_frames": len(sample["frame_phonemes"]),
        "num_ref_phonemes": len(true_phonemes),
        "num_pred_phonemes": len(pred_phonemes),
        "cer": metrics["cer"],
        "substitutions": metrics["substitutions"],
        "insertions": metrics["insertions"],
        "deletions": metrics["deletions"],
        "matched_boundaries": metrics["matched_boundaries"],
        "mean_boundary_error_ms": metrics["mean_boundary_error_ms"],
        "median_boundary_error_ms": metrics["median_boundary_error_ms"],
        "max_boundary_error_ms": metrics["max_boundary_error_ms"],
        "true_boundary_times_ms": " ".join(f"{value:.2f}" for value in true_boundary_times_ms[1:]),
        "pred_boundary_times_ms": " ".join(f"{value:.2f}" for value in pred_times_ms[1:]),
        "true_phonemes": " ".join(true_phonemes),
        "pred_phonemes": " ".join(pred_phonemes),
        "pred_stop_probs": " ".join(f"{value:.4f}" for value in stop_probs),
    }
    results.append(result_row)

    print(
        f"{sample['speaker']}/{sample['utterance_id']}: "
        f"CER={metrics['cer']:.4f}, "
        f"mean_boundary_error_ms={metrics['mean_boundary_error_ms']}"
    )

fieldnames = list(results[0].keys())
with open(OUTPUT_CSV, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f"Saved phrase-level evaluation to: {OUTPUT_CSV}")


## Summary

In [ ]:
mean_cer = sum(row["cer"] for row in results if row["cer"]<=1) / len(results)
valid_boundary_errors = [
    row["mean_boundary_error_ms"]
    for row in results
    if row["mean_boundary_error_ms"] is not None and row["cer"]<=1
]
mean_boundary_error = sum(valid_boundary_errors) / len(valid_boundary_errors) if valid_boundary_errors else None

print(f"Phrases evaluated: {len(results)}")
print(f"Mean CER: {mean_cer:.4f}")
print(f"Mean boundary error (ms): {mean_boundary_error}")

sorted(results, key=lambda row: row["cer"], reverse=True)[:5]


In [ ]:
# Phrases evaluated: 23324
# Mean CER: 0.1294
# Mean boundary error (ms): 38.27446933789227

In [ ]:
import pandas as pd
res = pd.DataFrame(results)
res

In [ ]:
import panphon.distance
from ToIPA import corpres2ipa
from tqdm import tqdm


dst = panphon.distance.Distance()
overall_pfer = []

for row in tqdm(results):
    ipa_pred = corpres2ipa(row['pred_phonemes'].split(' '))
    
    ipa_true = corpres2ipa(row['true_phonemes'].split(' '))
    true_len = len([s for s in ipa_true.split(' ') if s!=''])
    
    
    dist = dst.feature_edit_distance(ipa_pred, ipa_true)
    
    overall_pfer.append(dist/true_len)

In [ ]:
more1 = []
less1 = []
for p in overall_pfer:
    if p>1:
        more1.append(p)
    else:
        less1.append(p)
    

In [ ]:
print(sum(less1)/len(less1))
print(len(more1))

In [ ]:
res['PFER'] = overall_pfer

In [ ]:
res[res['PFER'] > 1]

In [ ]:
import yaml
import warnings

warnings.filterwarnings("ignore", message="calling yaml.load")